In [1]:
import folium, math
import numpy as np
import pandas as pd
import geopy.distance
import geopandas as gpd
import branca.colormap as cm
import folium.plugins as plugins
from tqdm import tqdm
tqdm.pandas()
# import matplotlib.pyplot as plt


In [2]:
homes_tao = pd.read_parquet("/glade/derecho/scratch/ggantos/CAT/home_locations_tao_nofilters/home_loc_by_row_group.parquet")
homes_org = pd.read_parquet("/glade/derecho/scratch/cbecker/CAT/home_locations/home_loc_by_row_group.parquet")
print(homes_tao.shape)
print(homes_org.shape)

(34603918, 8)
(57379828, 4)


In [3]:
survey_df = pd.read_csv("../Datasets/final_data/2024_AR_longitudinal_final.csv")
affected_zips = [str(int(x)) for x in survey_df["inputzip"].unique() if not math.isnan(x)]

In [4]:
homes_comp = homes_tao[homes_tao.zipcode.isin(affected_zips)]
print(homes_comp.shape)


(19121793, 8)


In [5]:
homes_org_latlng = homes_org[["lat", "lng"]].rename(columns={"lat": "lat_org", "lng": "lng_org"})
homes_comp = homes_comp.merge(homes_org_latlng, left_index=True, right_index=True, how='left')
print(homes_comp.shape)
homes_comp

(19121793, 10)


,duration_home,lat,lng,duration_work,lat_work,lng_work,zipcode,row_group,lat_org,lng_org
uid,,,,,,,,,,
00011729619ceba676e1f34f6de0b53bc3879baca7fffa9b0a0e70f466f492ba,7 days 19:34:19,34.244810,-118.270687,6 days 14:37:51,34.237289,-118.266098,91042,0,34.244709,-118.270622
0001167f185e739022211a8efa66527eb29198fd8748553010449184dcd797d2,39 days 15:30:56,34.071571,-118.083031,39 days 15:30:56,34.071571,-118.083031,91770,0,34.071499,-118.083023
0001156634b4a29559465f04454dfe7ebf57413a107e16c25194d1366969ccfc,16 days 01:56:30,34.219634,-118.987127,7 days 09:02:40,34.190399,-118.924202,93012,0,34.216339,-118.990433
000113ce3330cb882b58599914da21cdc3f0519a1c23811ef47349987eee7a26,26 days 12:06:56,34.173292,-118.346361,3 days 22:23:38,36.797901,-119.682055,91505,0,34.173294,-118.346397
000111f664ca4cbd348f563a25b5cdf91dca02e4394f0376d7f8db0145e4e5fe,13 days 04:51:54,33.711716,-117.962612,0 days 00:26:40,34.177036,-118.306351,92708,0,33.711700,-117.962601
...,...,...,...,...,...,...,...,...,...,...
ffff35886f9990e480c15e0fdc6a8e012d334999c825d859710c1015c1524c16,12 days 22:21:53,34.252342,-117.273335,0 days 00:13:04,34.249136,-117.273354,92407,57430,34.252392,-117.273392
ffff35886622a6a96b3801966dbe82be14a08be29bac815d234269d24749f3e1,7 days 00:45:04,34.031921,-117.327011,7 days 00:45:04,34.031921,-117.327011,92313,57430,34.031921,-117.327011
ffff355be60a1a7fc39a1566a4db424d97915054083e5f5cbf8c74710d188b32,7 days 01:10:59,33.725626,-117.911915,7 days 01:10:59,33.725626,-117.911915,92704,57430,33.725601,-117.911903


In [6]:
homes_comp["dist_home"] = homes_comp.progress_apply(lambda x: geopy.distance.distance((x.lat, x. lng), (x.lat_org, x. lng_org)).km, axis=1)

100%|██████████| 19121793/19121793 [21:17<00:00, 14964.89it/s]


In [7]:
homes_comp.sort_values("dist_home", ascending=False, inplace=True)
homes_comp

,duration_home,lat,lng,duration_work,lat_work,lng_work,zipcode,row_group,lat_org,lng_org,dist_home
uid,,,,,,,,,,,
0d449a0b33c1803ce62b6eab05ea696fc9008a77a9e2e37f6a9146ac18556757,12 days 20:49:27,32.701630,-116.355110,12 days 20:13:41,32.848270,-116.954502,91906,2975,41.747719,-124.202972,1220.652360
5e9bb9b0dd7ff2ab3750f0eb6c731bae27d635f50340edcde72f95fae7fe5a1c,10 days 03:33:20,32.721470,-117.000238,8 days 20:22:53,32.759635,-117.014324,91977,21226,41.774872,-124.187500,1188.978761
762448f599eee0a9f289b9325c52a3a2412dfb9c2ea2f90e9d8d611700c3c7d9,17 days 21:28:04,32.725800,-116.993301,2 days 00:00:29,41.784302,-124.130402,91977,26506,41.784302,-124.130402,1187.062187
608d3934640a53678e6d80817db96cd73812390b5b70a7582d58167b83001034,17 days 20:25:21,32.691729,-117.115134,5 days 11:18:54,41.781739,-124.133207,92113,21662,41.781700,-124.133202,1184.544577
54fec0df638fb73c4832663ee843cac9f82500a8663682c4d2ac6a67dcfc6173,9 days 09:28:20,32.804020,-117.109537,4 days 22:06:04,41.769368,-124.207129,92124,19072,41.769371,-124.207108,1176.333832
...,...,...,...,...,...,...,...,...,...,...,...
459637c0fd91104d0babd62c6b67a1faeb47956418c272518ccdf986a5fb7d5d,0 days 14:43:14,34.054401,-118.244003,0 days 14:43:14,34.054401,-118.244003,90012,15612,34.054401,-118.244003,0.000000
a364130d656118a7cdf4a47086939cadc9b0e59a2d223c1ce4f6b13025d19d0a,0 days 04:36:00,34.426392,-117.300880,0 days 04:36:00,34.426392,-117.300880,92345,36651,34.426392,-117.300880,0.000000
a36412f1c47ee0b5130252be41fd23d3cff6877764b4db55c6c48ffea5acf9ad,6 days 04:51:48,34.025101,-118.155602,1 days 21:57:12,34.084921,-118.294308,90022,36651,34.025101,-118.155602,0.000000


In [8]:
print(homes_comp["dist_home"].mean())
print(homes_comp["dist_home"].max())
print(homes_comp["dist_home"].min())

18.595992433694626
1220.6523598970368
0.0


In [10]:
homes_comp["dist_work"] = homes_comp.progress_apply(lambda x: geopy.distance.distance((x.lat_work, x. lng_work), (x.lat_org, x. lng_org)).km, axis=1)
homes_comp = gdf.

100%|██████████| 19121793/19121793 [21:54<00:00, 14552.10it/s]


In [11]:
print(homes_comp["dist_work"].mean())
print(homes_comp["dist_work"].max())
print(homes_comp["dist_work"].min())

24.524036839145055
1216.1638745622479
0.0


In [2]:
# homes_comp.to_parquet("/glade/derecho/scratch/ggantos/CAT/home_locations_tao_nofilters/home_loc_by_row_group_comp.parquet")
homes_comp = pd.read_parquet("/glade/derecho/scratch/ggantos/CAT/home_locations_tao_nofilters/home_loc_by_row_group_comp.parquet")

In [3]:
bins = np.arange(0.0, 1400.0, 100.0)
homes_comp["dist_home_bin"] = pd.cut(homes_comp.dist_home, bins)
homes_comp["dist_work_bin"] = pd.cut(homes_comp.dist_work, bins)
homes_comp["dist_home_bin"]

uid
0d449a0b33c1803ce62b6eab05ea696fc9008a77a9e2e37f6a9146ac18556757    (1200.0, 1300.0]
5e9bb9b0dd7ff2ab3750f0eb6c731bae27d635f50340edcde72f95fae7fe5a1c    (1100.0, 1200.0]
762448f599eee0a9f289b9325c52a3a2412dfb9c2ea2f90e9d8d611700c3c7d9    (1100.0, 1200.0]
608d3934640a53678e6d80817db96cd73812390b5b70a7582d58167b83001034    (1100.0, 1200.0]
54fec0df638fb73c4832663ee843cac9f82500a8663682c4d2ac6a67dcfc6173    (1100.0, 1200.0]
                                                                          ...       
459637c0fd91104d0babd62c6b67a1faeb47956418c272518ccdf986a5fb7d5d                 NaN
a364130d656118a7cdf4a47086939cadc9b0e59a2d223c1ce4f6b13025d19d0a                 NaN
a36412f1c47ee0b5130252be41fd23d3cff6877764b4db55c6c48ffea5acf9ad                 NaN
a36412c91a01d3192f75e7ebe8bb4eab0cd680eff1bf40f71ef3a13decf36aec                 NaN
a364363528582665ecf0862e159f925dd1ae2d2271c3169d4598fe85ea9b43ed                 NaN
Name: dist_home_bin, Length: 19121793, dtype: category
Catego

In [4]:
cols_grpby = homes_comp.columns.drop(['zipcode', 'row_group', 'dist_work_bin'])
homes_grouped = homes_comp[cols_grpby].groupby("dist_home_bin").mean()
homes_grouped

/glade/derecho/scratch/ggantos/tmp/ipykernel_124059/1584268503.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  homes_grouped = homes_comp[cols_grpby].groupby("dist_home_bin").mean()


,duration_home,lat,lng,duration_work,lat_work,lng_work,lat_org,lng_org,dist_home,dist_work
dist_home_bin,,,,,,,,,,
"(0.0, 100.0]",16 days 16:59:19.973458984,34.156747,-118.081483,8 days 08:13:24.124998467,34.197399,-118.120610,34.150246,-118.080803,7.560213,19.236823
"(100.0, 200.0]",13 days 00:50:38.217932421,34.099255,-117.931160,6 days 03:30:59.862490614,34.132936,-117.975584,33.908680,-117.817650,143.005594,79.053649
"(200.0, 300.0]",12 days 00:50:51.842660828,35.073660,-118.867332,5 days 16:19:19.079720119,35.334376,-119.106622,35.223095,-118.987885,240.649758,133.816889
"(300.0, 400.0]",10 days 02:25:43.808971130,35.448569,-119.124490,4 days 18:08:40.739552724,35.650245,-119.245669,35.154537,-118.795654,348.266280,204.603768
"(400.0, 500.0]",11 days 12:00:42.660039094,34.942597,-119.003241,5 days 12:10:54.620219322,35.989933,-119.940522,36.276836,-120.121834,454.487216,221.124224
"(500.0, 600.0]",11 days 00:52:20.048915327,34.314188,-118.285001,5 days 04:55:22.767516762,36.273331,-120.123289,37.322639,-121.184029,545.869086,249.412474
"(600.0, 700.0]",11 days 20:11:28.705100755,33.837119,-117.769611,5 days 12:24:04.242005015,36.156737,-119.850921,38.038124,-121.497887,635.677064,305.797093
"(700.0, 800.0]",9 days 18:59:31.472724369,33.104700,-117.271750,4 days 17:24:03.985213196,35.492588,-119.319303,38.492523,-121.790527,740.588142,418.133870
"(800.0, 900.0]",10 days 12:18:59.769228848,33.636972,-117.511021,5 days 06:36:50.115937586,36.590918,-119.708473,40.075157,-122.329727,838.416994,460.575282


In [ ]:
vmax = math.ceil(homes_comp["dist_home"].max() / 10.0) * 10
linear = cm.LinearColormap(["b", "r"],
                           caption= "Distance between TAO and SKM home location",
                           vmin=1,
                           vmax=vmax,
                          ).to_step(index=bins,
                                    )

m = folium.Map(location=[36.0549, -118.2426],
               zoom_start=6,
               control_scale=True,
               )

for dist_bin, row in homes_grouped.iterrows():

    fg = folium.FeatureGroup(name=str(dist_bin)).add_to(m)
    folium.PolyLine(
        locations=[row.loc[["lat", "lng"]].to_list(), row.loc[["lat_org", "lng_org"]].to_list()],
        color=linear(row["dist_home"]),
        weight=4,
        highlight_function=lambda feature: {
            "color": "#78e05b",
            "weight": 6,
        },
        tooltip=row[["dist_home", "dist_work", "duration_home", "duration_work"]],
        ).add_to(fg)

linear.add_to(m)
plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
).add_to(m)
folium.LayerControl().add_to(m)

m.save("html/home_deltas.html")
m

![title](img/home_dist_binned_avg.png)

In [ ]:
## too weighty for rendering or saving out, shifting to another track
for dist_bin, row in homes_grouped.iterrows():
    linear = cm.LinearColormap(["b", "r"],
                               caption= "Distance between TAO and SKM home location",
                               vmin=dist_bin.left,
                               vmax=dist_bin.right,
                              ).to_step(10)
    
    m = folium.Map(location=[36.0549, -118.2426],
                   zoom_start=6,
                   control_scale=True,
                   )
    
    bins = np.arange(dist_bin.left, dist_bin.right, 10)
    for name, group in homes_comp[homes_comp.dist_home_bin == dist_bin].groupby([pd.cut(homes_comp.dist_home, bins)]):
        fg = folium.FeatureGroup(name=str(name)).add_to(m)
        for i, row in group.iterrows():
            folium.PolyLine(
                locations=[row.loc[["lat", "lng"]].to_list(), row.loc[["lat_org", "lng_org"]].to_list()],
                color=linear(row["dist_home"]),
                weight=4,
                highlight_function=lambda feature: {
                    "color": "#78e05b",
                    "weight": 6,
                },
                tooltip=row[["dist_home", "dist_work", "duration_home", "duration_work"]],
                ).add_to(fg)

    linear.add_to(m)
    plugins.Fullscreen(
        position="topright",
        title="Expand me",
        title_cancel="Exit me",
        force_separate_button=True,
    ).add_to(m)
    folium.LayerControl().add_to(m)
    
    m.save(f"html/home_deltas_{str(dist_bin)}.html")


/glade/derecho/scratch/ggantos/tmp/ipykernel_24132/1271525601.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for name, group in homes_comp[homes_comp.dist_home_bin == dist_bin].groupby([pd.cut(homes_comp.dist_home, bins)]):
